# DeepGBoost — Paper Benchmark

Reproduces the regression benchmark from the original DGBF paper:

> Delgado-Panadero et al. (2023). *A generalized decision tree ensemble based on the NeuralNetworks architecture: Distributed Gradient Boosting Forest (DGBF)*. Applied Intelligence, 53, 22991–23003.

**Methodology** — 200 randomised 80/20 train-test splits per dataset, metric: R².

**Datasets** — 9 regression datasets from the UCI Machine Learning Repository:
Parkinson, Wine, Concrete, Obesity, NavalVessel, Temperature, Cargo2000, BikeSales, Superconduct.

**Models**
- `sklearn` GradientBoostingRegressor  (GBDT)
- `sklearn` RandomForestRegressor  (RF)
- `deepgboost` DeepGBoostRegressor  (DGBF)

In [ ]:
import io
import zipfile
import warnings
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from deepgboost import DeepGBoostRegressor

warnings.filterwarnings("ignore")
DATA_DIR    = Path("data")
RESULTS_DIR = Path(".")
DATA_DIR.mkdir(exist_ok=True)

N_RUNS    = 200
TEST_SIZE = 0.2
RNG       = 42
print("Libraries loaded.")

---
## Datasets

Downloads each dataset on first run and caches it locally under `data/`.

In [ ]:
DATASETS = [
    {
        "name": "Parkinson",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/telemonitoring/parkinsons_updrs.data",
        "file": DATA_DIR / "parkinson.csv",
        "sep": ",",
    },
    {
        "name": "Wine",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        "file": DATA_DIR / "wine_red.csv",
        "sep": ";",
    },
    {
        "name": "Concrete",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls",
        "file": DATA_DIR / "concrete.csv",
        "sep": ",",
        "excel": True,
    },
    {
        "name": "NavalVessel",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00316/UCI%20CBM%20Dataset.zip",
        "file": DATA_DIR / "UCI_CBM_Dataset.csv",
        "sep": "\t",
        "zip": "UCI CBM Dataset/UCI_CBM_Dataset.txt",
    },
    {
        "name": "Temperature",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00514/Bias_correction_ucl.csv",
        "file": DATA_DIR / "Bias_correction_ucl.csv",
        "sep": ",",
    },
    {
        "name": "Cargo2000",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00382/c2k_data_comma.csv",
        "file": DATA_DIR / "c2k_data_comma.csv",
        "sep": ",",
    },
    {
        "name": "BikeSales",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00560/SeoulBikeData.csv",
        "file": DATA_DIR / "SeoulBikeData.csv",
        "sep": ",",
        "encoding": "unicode_escape",
    },
    {
        "name": "Superconduct",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00464/superconduct.zip",
        "file": DATA_DIR / "superconduct.csv",
        "sep": ",",
        "zip": "train.csv",
    },
    {
        "name": "Obesity",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00544/ObesityDataSet_raw_and_data_sinthetic.csv",
        "file": DATA_DIR / "ObesityDataSet.csv",
        "sep": ",",
    },
]


def _load_dataset(cfg: dict) -> tuple[np.ndarray, np.ndarray] | None:
    """Download (if needed) and return (X, y) as numeric numpy arrays."""
    path: Path = cfg["file"]

    if not path.exists():
        print(f"  Downloading {cfg['name']} ...", end="", flush=True)
        try:
            raw = urllib.request.urlopen(cfg["url"], timeout=30).read()
        except Exception as exc:
            print(f" FAILED ({exc})")
            return None

        if cfg.get("zip"):
            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                raw = zf.read(cfg["zip"])

        if cfg.get("excel"):
            df = pd.read_excel(io.BytesIO(raw))
        else:
            df = pd.read_csv(
                io.StringIO(raw.decode(cfg.get("encoding", "utf-8"), errors="replace")),
                sep=cfg["sep"],
            )

        df.to_csv(path, index=False)
        print(" done")

    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number]).dropna()
    if df.shape[1] < 2:
        return None
    X = df.iloc[:, :-1].values.astype(float)
    y = df.iloc[:, -1].values.astype(float)
    return X, y


datasets: dict[str, tuple[np.ndarray, np.ndarray]] = {}
for cfg in DATASETS:
    result = _load_dataset(cfg)
    if result is not None:
        X, y = result
        datasets[cfg["name"]] = (X, y)
        print(f"  {cfg['name']:15s}  shape={X.shape}")
    else:
        print(f"  {cfg['name']:15s}  SKIPPED")

---
## Models

In [ ]:
MODELS = {
    "GBDT": GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1),
    "RF":   RandomForestRegressor(n_estimators=100, max_depth=8),
    "DGBF": DeepGBoostRegressor(n_trees=10, n_layers=15, max_depth=4, learning_rate=0.1),
}

---
## Bootstrap benchmark

200 randomised 80/20 splits per dataset — same methodology as the paper.

In [ ]:
def run_bootstrap(
    models: dict,
    X: np.ndarray,
    y: np.ndarray,
    n_runs: int = N_RUNS,
    test_size: float = TEST_SIZE,
) -> dict[str, np.ndarray]:
    """Return dict of model_name -> R² scores array (length n_runs)."""
    scores = {name: np.zeros(n_runs) for name in models}

    for i in tqdm(range(n_runs), leave=False):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=i
        )
        for name, model in models.items():
            y_pred = model.fit(X_train, y_train).predict(X_test)
            scores[name][i] = r2_score(y_test, y_pred)

    return scores


all_scores: dict[str, dict[str, np.ndarray]] = {}

for ds_name, (X, y) in datasets.items():
    print(f"Running {ds_name} ...", flush=True)
    all_scores[ds_name] = run_bootstrap(MODELS, X, y)
    means = {m: f"{v.mean():.4f}" for m, v in all_scores[ds_name].items()}
    print(f"  {means}")

print("\nDone.")

---
## Results — R² score distributions per dataset

In [ ]:
n_datasets = len(all_scores)
ncols = 3
nrows = (n_datasets + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

colors = {"GBDT": "steelblue", "RF": "darkorange", "DGBF": "seagreen"}

for ax, (ds_name, scores) in zip(axes, all_scores.items()):
    all_vals = np.concatenate(list(scores.values()))
    bins = np.linspace(all_vals.min(), all_vals.max(), 40)
    for model_name, model_scores in scores.items():
        ax.hist(model_scores, bins=bins, alpha=0.55, label=model_name,
                color=colors[model_name], edgecolor="none")
    ax.set_title(ds_name, fontsize=11)
    ax.set_xlabel("R²")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

for ax in axes[n_datasets:]:
    ax.set_visible(False)

plt.suptitle(f"R² Score Distribution — {N_RUNS} bootstrap runs per dataset",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "regression_benchmark.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Results — Mean R² per dataset (summary bar chart)

In [ ]:
ds_names   = list(all_scores.keys())
model_names = list(MODELS.keys())
x = np.arange(len(ds_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
offsets = np.linspace(-(len(model_names) - 1) / 2, (len(model_names) - 1) / 2, len(model_names))

for offset, model_name in zip(offsets, model_names):
    means = [all_scores[ds][model_name].mean() for ds in ds_names]
    stds  = [all_scores[ds][model_name].std()  for ds in ds_names]
    ax.bar(x + offset * width, means, width, yerr=stds, capsize=3,
           label=model_name, color=colors[model_name], alpha=0.85,
           edgecolor="k", linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels(ds_names, rotation=20, ha="right")
ax.set_ylabel("Mean R² (higher is better)")
ax.set_title(f"Mean R² per dataset — {N_RUNS} bootstrap runs")
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "benchmark_summary.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Save results to `results.md`

In [ ]:
import platform
import sklearn
import deepgboost as dgb

# Build summary table
rows = []
for ds_name in ds_names:
    row = {"Dataset": ds_name}
    for model_name in model_names:
        s = all_scores[ds_name][model_name]
        row[f"{model_name} mean"] = round(float(s.mean()), 4)
        row[f"{model_name} std"]  = round(float(s.std()),  4)
        row[f"{model_name} wins"] = int(
            (all_scores[ds_name]["DGBF"] > all_scores[ds_name][model_name]).mean() * 100
        ) if model_name != "DGBF" else "-"
    rows.append(row)

df = pd.DataFrame(rows)

def df_to_markdown(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep    = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows_md = ["| " + " | ".join(str(r[c]) for c in cols) + " |" for _, r in df.iterrows()]
    return "\n".join([header, sep] + rows_md)

wins_vs_gbdt = sum(
    all_scores[ds]["DGBF"].mean() > all_scores[ds]["GBDT"].mean()
    for ds in ds_names
)
wins_vs_rf = sum(
    all_scores[ds]["DGBF"].mean() > all_scores[ds]["RF"].mean()
    for ds in ds_names
)

md = f"""# DeepGBoost Benchmark Results

Reproduces the regression benchmark from the DGBF paper.
All R² scores are means over **{N_RUNS} randomised 80/20 bootstrap splits**.

## Environment

- Python {platform.python_version()}
- deepgboost=={dgb.__version__}
- scikit-learn=={sklearn.__version__}
- numpy=={np.__version__}

## Summary

DGBF surpasses GBDT in **{wins_vs_gbdt}/{len(ds_names)} datasets** and RF in **{wins_vs_rf}/{len(ds_names)} datasets**.

## Results

{df_to_markdown(df)}

Columns `GBDT wins` and `RF wins` show the % of runs where DGBF outperformed the baseline.

![Distribution per dataset](regression_benchmark.png)

![Summary](benchmark_summary.png)
"""

out_path = RESULTS_DIR / "results.md"
out_path.write_text(md, encoding="utf-8")
print(f"Saved to {out_path.resolve()}")
print(md)